# PlantID — Final Model (`Model_Final.ipynb`)

**PlantID** is a mobile app: a user photographs a plant, the model below names the species, and the app shows
care tips for it. This notebook contains **every step of building the model**, from collecting photos to the
file that runs on the phone.

### What changed since `Model_Base.ipynb`

| | Base model (CAT 2) | Final model (this notebook) |
|---|---|---|
| Plants recognised | 7 species | **30 species** + an **"other plant"** answer for anything else |
| Photos | 3,446 photos, one campus in India | **28,969** photos from Plant100-CLS and iNaturalist (≈19,600 different photographers) |
| Test result | 100% accuracy, but not trustworthy (see below) | Measured on photos from people the model never learned from |
| Phone file | none | TensorFlow Lite file, checked against the original model |

**Why we had to collect new data and train again:** adding 23 species means the model needs examples of each
new plant and a new output layer, so the model is retrained from the start. While doing this we also found
two problems in the base model's data:
1. Most Plant100 classes were photographed in **a single session of under an hour, with photos only seconds
   apart**, so most photos of a class are near-copies of the same few plants. With a random split, the test
   photos were almost identical to training photos, which is why the base model scored 100%.
2. Plant100's **"Ficus elastica" (rubber plant) folder actually contains weeping figs** (*Ficus benjamina*).

Both are fixed in this notebook (Sections 3–4).

### How to run
* **Kaggle (recommended):** add the `plantid-v1` dataset as input, turn on a GPU and Internet, then
  *Save Version → Save & Run All*. About 3–5 hours on a T4 GPU.
* **Google Colab:** put `plantid_v1.zip` in `MyDrive/PlantID/`, select a GPU runtime, *Run all*.
* By default the notebook **loads the prepared dataset**. Sections 3–4 contain the full collection and cleaning
  code; set `RUN_DATA_COLLECTION` / `RUN_DATASET_BUILD` to `True` to rebuild it from scratch (≈6.5 GB download).

### Contents
1. Problem and approach (theory in plain words)
2. Setup
3. Data collection
4. Data cleaning and splitting
5. Exploratory data analysis (observations)
6. Preparing images for the model
7. Base model, retrained on 30 species, and cross-validation
8. Comparison models
9. Choosing and evaluating the final model
10. Unknown plants and invalid photos
11. Explainability: what the model looked at
12. Export for Android (TensorFlow Lite) and verification
13. Demo: the engine as the app uses it
14. Saving the deliverables, conclusion and limitations

## 1. Problem and approach

**Task.** Given one photo, choose which of 31 answers it is (30 species or "other plant"). Learning from photos
that already have the correct name attached is called **supervised learning**, and choosing one answer from a
fixed list is **classification**.

**Model type: convolutional neural network (CNN).** A CNN looks at an image through many small filters. Early
filters detect edges and colours, later ones combine them into shapes such as leaf outlines, vein patterns and
flowers. The last layer turns these patterns into a score for every class; **softmax** converts the scores into
probabilities that add up to 1.

**Transfer learning.** Training a CNN from nothing needs millions of photos. Instead we start from a network
already trained on ImageNet (1.2 million everyday photos), which already knows edges, textures and shapes, and
only teach it our plants:
1. **Phase 1:** keep the pretrained network frozen and train only the new final layer (fast).
2. **Phase 2, fine-tuning:** unfreeze the network and adjust all of it slightly with a small learning rate.

**Why phone-sized networks.** The model must run on an Android phone, so we compare networks designed for
phones: **MobileNetV2** (our base model), **MobileNetV3-Large** and **EfficientNet-B0**. We also train a small
CNN from scratch and a simple linear classifier to show how much transfer learning helps.

**Checking the results are real.** A model can look good by memorising. We prevent this with a strict split
(photos of the same plant or by the same photographer are never in both training and test), cross-validation
of the base model, and tests on plant species the model has never seen.

## 2. Setup

In [ ]:
import os, sys, glob, json, time, math, random, shutil, gc, zipfile, platform, subprocess, re, io, csv
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = os.path.exists("/kaggle/input") and not IN_COLAB
ENV = "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local"

# Optional steps (off by default): rebuild the dataset from the original sources.
RUN_DATA_COLLECTION = os.environ.get("PLANTID_COLLECT", "0") == "1"   # download raw photos (~6.5 GB, ~2 h)
RUN_DATASET_BUILD = RUN_DATA_COLLECTION or os.environ.get("PLANTID_BUILD", "0") == "1"   # clean + split (~15 min)
QUICK_RUN = os.environ.get("PLANTID_QUICK", "0") == "1"   # tiny smoke test of every cell; numbers meaningless

WORK_DIR = os.environ.get("PLANTID_WORK") or {"kaggle": "/kaggle/working/plantid_work",
                                              "colab": "/content/plantid_work"}.get(ENV, "C:/PlantID_data")
OUT_DIR = os.environ.get("PLANTID_OUT") or {"kaggle": "/kaggle/working/plantid_outputs",
                                            "colab": "/content/drive/MyDrive/PlantID/outputs"}.get(ENV, "./outputs")
if IN_COLAB:                                  # mount Google Drive before creating folders on it
    from google.colab import drive
    drive.mount("/content/drive")
for sub in ["models", "logs", "figures", "export", "final"]:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

for pkg, mod in [("imagehash", "imagehash"), ("ai-edge-litert", "ai_edge_litert")]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw
import imagehash
import tensorflow as tf
import keras
from keras import layers
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_fscore_support, top_k_accuracy_score)
try:
    from ai_edge_litert.interpreter import Interpreter as TFLiteInterpreter
except ImportError:
    TFLiteInterpreter = tf.lite.Interpreter

SEED = 42
keras.utils.set_random_seed(SEED)       # same random choices every run
gpus = tf.config.list_physical_devices("GPU")
USE_MIXED = False
if gpus:
    cc = tf.config.experimental.get_device_details(gpus[0]).get("compute_capability", (0, 0))
    USE_MIXED = cc[0] >= 7               # float16 speeds up T4/A100 GPUs
    keras.mixed_precision.set_global_policy("mixed_float16" if USE_MIXED else "float32")
print(f"Environment: {ENV} | TensorFlow {tf.__version__} | Keras {keras.__version__} | GPU: {[g.name for g in gpus] or 'none'}")
print(f"RUN_DATA_COLLECTION={RUN_DATA_COLLECTION} RUN_DATASET_BUILD={RUN_DATASET_BUILD} QUICK_RUN={QUICK_RUN}")
print("Outputs:", OUT_DIR)

### Plant list and data sources

All 30 species, their scientific names and where their photos come from. `taxon` is the name searched on
iNaturalist; `p100` lists the Plant100-CLS folders used. The `other_plant` list holds look-alikes and other
common plants (trained as "other plant"); the `openset` list holds species that are **never** trained on and
are only used to test how the model reacts to unfamiliar plants.

In [ ]:
CONFIG = {
 "_doc": "Single source of truth for PlantID v1 classes and data sources. Class output order = alphabetical by slug (see labels.txt produced by build_dataset.py). iNat 'taxon' is the iNaturalist taxon name searched; 'taxon_id' pins it. 'p100' lists Plant100-CLS folder names.",
 "version": "plantid_v1",
 "inat_licenses": [
  "cc0",
  "cc-by",
  "cc-by-sa",
  "cc-by-nc",
  "cc-by-nc-sa"
 ],
 "targets": {
  "_doc": "Most Plant100 classes were shot in one short session (photos seconds apart; a few span 2-4 days), so each folder is treated as ONE group, used for TRAINING ONLY and capped low. Validation/test come from iNaturalist only.",
  "class_total": 900,
  "class_p100_max": 150,
  "inat_user_cap_class": 8,
  "inat_user_cap_aux": 3,
  "other_inat_per_species": 55,
  "other_p100_per_class": 5,
  "openset_inat_per_species": 70,
  "openset_p100_per_class": 10
 },
 "classes": [
  {
   "slug": "aloe_vera",
   "common": "Aloe vera",
   "scientific": "Aloe vera",
   "category": "herb_edible",
   "taxon": "Aloe vera",
   "taxon_id": 126882
  },
  {
   "slug": "anthurium",
   "common": "Anthurium",
   "scientific": "Anthurium andraeanum",
   "category": "indoor",
   "taxon": "Anthurium andraeanum",
   "taxon_id": 69235
  },
  {
   "slug": "areca_palm",
   "common": "Areca palm",
   "scientific": "Dypsis lutescens",
   "category": "indoor",
   "taxon": "Chrysalidocarpus lutescens",
   "taxon_id": 276548,
   "p100": [
    "Areca_palm"
   ]
  },
  {
   "slug": "arrowhead_vine",
   "common": "Arrowhead vine",
   "scientific": "Syngonium podophyllum",
   "category": "indoor",
   "taxon": "Syngonium podophyllum",
   "taxon_id": 143208
  },
  {
   "slug": "basil",
   "common": "Basil",
   "scientific": "Ocimum basilicum",
   "category": "herb_edible",
   "taxon": "Ocimum basilicum",
   "taxon_id": 61398
  },
  {
   "slug": "bougainvillea",
   "common": "Bougainvillea",
   "scientific": "Bougainvillea spp.",
   "category": "garden",
   "taxon": "Bougainvillea",
   "taxon_id": 53804,
   "p100": [
    "Bougainvillea_glabra"
   ]
  },
  {
   "slug": "canna_lily",
   "common": "Canna lily",
   "scientific": "Canna indica",
   "category": "garden",
   "taxon": "Canna indica",
   "taxon_id": 146643
  },
  {
   "slug": "chilli_pepper",
   "common": "Chilli / sweet pepper",
   "scientific": "Capsicum annuum",
   "category": "herb_edible",
   "taxon": "Capsicum annuum",
   "taxon_id": 48514
  },
  {
   "slug": "coleus",
   "common": "Coleus",
   "scientific": "Coleus scutellarioides",
   "category": "garden",
   "taxon": "Coleus scutellarioides",
   "taxon_id": 291284,
   "p100": [
    "Coleus_scutellarioides"
   ]
  },
  {
   "slug": "corn_plant",
   "common": "Corn plant",
   "scientific": "Dracaena fragrans",
   "category": "indoor",
   "taxon": "Dracaena fragrans",
   "taxon_id": 126515
  },
  {
   "slug": "croton",
   "common": "Croton",
   "scientific": "Codiaeum variegatum",
   "category": "garden",
   "taxon": "Codiaeum variegatum",
   "taxon_id": 127387
  },
  {
   "slug": "crown_of_thorns",
   "common": "Crown of thorns",
   "scientific": "Euphorbia milii",
   "category": "garden",
   "taxon": "Euphorbia milii",
   "taxon_id": 162839,
   "p100": [
    "Euphorbia_milli"
   ]
  },
  {
   "slug": "dumb_cane",
   "common": "Dumb cane",
   "scientific": "Dieffenbachia seguine",
   "category": "indoor",
   "taxon": "Dieffenbachia seguine",
   "taxon_id": 206266
  },
  {
   "slug": "duranta",
   "common": "Duranta",
   "scientific": "Duranta erecta",
   "category": "garden",
   "taxon": "Duranta erecta",
   "taxon_id": 162153,
   "p100": [
    "Duranta_erecta"
   ]
  },
  {
   "slug": "frangipani",
   "common": "Frangipani",
   "scientific": "Plumeria spp.",
   "category": "garden",
   "taxon": "Plumeria",
   "taxon_id": 48319,
   "p100": [
    "Plumeria",
    "Plumeria_pudica"
   ]
  },
  {
   "slug": "golden_pothos",
   "common": "Golden pothos / Money plant",
   "scientific": "Epipremnum aureum",
   "category": "indoor",
   "taxon": "Epipremnum aureum",
   "taxon_id": 69223,
   "p100": [
    "Epipremnum_aureum"
   ]
  },
  {
   "slug": "hibiscus",
   "common": "Hibiscus",
   "scientific": "Hibiscus rosa-sinensis",
   "category": "garden",
   "taxon": "Hibiscus × rosa-sinensis",
   "taxon_id": 1628098,
   "p100": [
    "Hibiscus_rosa_sinensis"
   ]
  },
  {
   "slug": "jade_plant",
   "common": "Jade plant",
   "scientific": "Crassula ovata",
   "category": "indoor",
   "taxon": "Crassula ovata",
   "taxon_id": 76465,
   "p100": [
    "Crassula_ovata"
   ]
  },
  {
   "slug": "ming_aralia",
   "common": "Ming aralia",
   "scientific": "Polyscias fruticosa",
   "category": "indoor",
   "taxon": "Polyscias fruticosa",
   "taxon_id": 284560,
   "p100": [
    "Polyscias_fruticosa"
   ]
  },
  {
   "slug": "mint",
   "common": "Mint",
   "scientific": "Mentha spp.",
   "category": "herb_edible",
   "taxon": "Mentha",
   "taxon_id": 54570
  },
  {
   "slug": "monstera",
   "common": "Monstera",
   "scientific": "Monstera deliciosa",
   "category": "indoor",
   "taxon": "Monstera deliciosa",
   "taxon_id": 122521
  },
  {
   "slug": "peace_lily",
   "common": "Peace lily",
   "scientific": "Spathiphyllum spp.",
   "category": "indoor",
   "taxon": "Spathiphyllum",
   "taxon_id": 206271
  },
  {
   "slug": "periwinkle",
   "common": "Madagascar periwinkle",
   "scientific": "Catharanthus roseus",
   "category": "garden",
   "taxon": "Catharanthus roseus",
   "taxon_id": 62924,
   "p100": [
    "Catharanthus_roseus"
   ]
  },
  {
   "slug": "rosemary",
   "common": "Rosemary",
   "scientific": "Salvia rosmarinus",
   "category": "herb_edible",
   "taxon": "Rosmarinus",
   "taxon_id": 82927
  },
  {
   "slug": "rubber_plant",
   "common": "Rubber plant",
   "scientific": "Ficus elastica",
   "category": "indoor",
   "taxon": "Ficus elastica",
   "taxon_id": 126507,
   "p100_note": "Plant100 'Ficus_elastica' folder was checked visually (2026-11-21): it contains green Ficus benjamina, not rubber plants. It is used for weeping_fig instead; rubber_plant is iNaturalist-only."
  },
  {
   "slug": "snake_plant",
   "common": "Snake plant",
   "scientific": "Dracaena trifasciata",
   "category": "indoor",
   "taxon": "Sansevieria trifasciata",
   "taxon_id": 67710,
   "p100": [
    "Sansevieria_trifasciata"
   ]
  },
  {
   "slug": "spider_plant",
   "common": "Spider plant",
   "scientific": "Chlorophytum comosum",
   "category": "indoor",
   "taxon": "Chlorophytum comosum",
   "taxon_id": 322492
  },
  {
   "slug": "tomato",
   "common": "Tomato",
   "scientific": "Solanum lycopersicum",
   "category": "herb_edible",
   "taxon": "Solanum lycopersicum",
   "taxon_id": 51737
  },
  {
   "slug": "weeping_fig",
   "common": "Weeping fig",
   "scientific": "Ficus benjamina",
   "category": "indoor",
   "taxon": "Ficus benjamina",
   "taxon_id": 50996,
   "p100": [
    "Ficus_benjamina",
    "Ficus_elastica"
   ],
   "p100_note": "Plant100 Ficus_benjamina = variegated form only; Plant100 Ficus_elastica = mislabeled green F. benjamina. Both used here so the class covers both forms."
  },
  {
   "slug": "zz_plant",
   "common": "ZZ plant",
   "scientific": "Zamioculcas zamiifolia",
   "category": "indoor",
   "taxon": "Zamioculcas zamiifolia",
   "taxon_id": 367277
  }
 ],
 "other_plant": {
  "_doc": "Trained as class 'other_plant'. Look-alikes of supported classes plus common plants. Mixed iNat + Plant100 so 'other' cannot be learned as 'Plant100 style'.",
  "inat": [
   {
    "taxon": "Philodendron hederaceum"
   },
   {
    "taxon": "Scindapsus pictus"
   },
   {
    "taxon": "Ficus lyrata"
   },
   {
    "taxon": "Ficus microcarpa"
   },
   {
    "taxon": "Aglaonema"
   },
   {
    "taxon": "Aloe",
    "without": [
     "Aloe vera"
    ]
   },
   {
    "taxon": "Agave americana"
   },
   {
    "taxon": "Lavandula"
   },
   {
    "taxon": "Solanum tuberosum"
   },
   {
    "taxon": "Solanum melongena"
   },
   {
    "taxon": "Melissa officinalis"
   },
   {
    "taxon": "Ocimum tenuiflorum"
   },
   {
    "taxon": "Chamaedorea elegans"
   },
   {
    "taxon": "Phoenix roebelenii"
   },
   {
    "taxon": "Impatiens walleriana"
   },
   {
    "taxon": "Heliconia"
   },
   {
    "taxon": "Tradescantia zebrina"
   },
   {
    "taxon": "Hedera helix"
   },
   {
    "taxon": "Lantana camara"
   },
   {
    "taxon": "Dracaena reflexa"
   },
   {
    "taxon": "Cordyline fruticosa"
   },
   {
    "taxon": "Acalypha wilkesiana"
   },
   {
    "taxon": "Portulacaria afra"
   },
   {
    "taxon": "Thaumatophyllum bipinnatifidum"
   },
   {
    "taxon": "Monstera adansonii"
   },
   {
    "taxon": "Peperomia obtusifolia"
   },
   {
    "taxon": "Musa"
   }
  ],
  "p100": "ALL_REMAINING"
 },
 "openset": {
  "_doc": "NEVER used in training. Species absent from all training classes, used only to measure unknown-plant rejection.",
  "inat": [
   {
    "taxon": "Streptocarpus ionanthus"
   },
   {
    "taxon": "Pilea peperomioides"
   },
   {
    "taxon": "Goeppertia"
   },
   {
    "taxon": "Nephrolepis exaltata"
   },
   {
    "taxon": "Begonia"
   },
   {
    "taxon": "Euphorbia pulcherrima"
   },
   {
    "taxon": "Brassica oleracea"
   },
   {
    "taxon": "Coriandrum sativum"
   },
   {
    "taxon": "Phalaenopsis"
   },
   {
    "taxon": "Zea mays"
   },
   {
    "taxon": "Kalanchoe blossfeldiana"
   },
   {
    "taxon": "Hoya carnosa"
   },
   {
    "taxon": "Pelargonium"
   }
  ],
  "p100": [
   "Azadirachta_indica",
   "Mangifera_indica",
   "Psidium_guajava",
   "Tecoma_stans",
   "Genus_rosa",
   "Chrysanthemum",
   "Tagetes",
   "Jasminum_sambac",
   "Ixora_coccinea",
   "Opuntia_ficus_indica",
   "Ricinus_communis",
   "Datura",
   "Gossypium",
   "Sorghum_bicolor",
   "Punica_granatum",
   "Calotropis_gigantea",
   "Lawsonia_inermis",
   "Murraya_koenigii",
   "Citrus_sinensis",
   "Mirabilis_jalapa"
  ]
 }
}

CLASS_META = {c["slug"]: c for c in CONFIG["classes"]}
print(len(CONFIG["classes"]), "species +", "other_plant;", len(CONFIG["openset"]["inat"]) + len(CONFIG["openset"]["p100"]), "open-set species")
pd.DataFrame(CONFIG["classes"])[["slug", "common", "scientific", "category"]]

## 3. Data collection

**Sources.**
* **Plant100-CLS** (Hugging Face, Apache-2.0 licence): the dataset used by the base model. Phone photos from one
  campus in India.
* **iNaturalist**: a citizen-science site where people upload plant photos and the community checks the name.
  We use only openly licensed photos (CC0 / CC BY / CC BY-SA / CC BY-NC / CC BY-NC-SA); every photo's licence and
  photographer are recorded in `metadata.csv`.

**Quality rules for iNaturalist photos**
* at least one other person agreed with the plant's name, and nobody disagreed;
* only the first photo of each observation (one plant = one photo);
* at most 8 photos per photographer per species (variety, and less risk of one home dominating).

Downloading takes about two hours, so this step is **off by default** and the notebook uses the prepared dataset.

In [ ]:
API = "https://api.inaturalist.org/v1/"
UA = {"User-Agent": "plantid-dataset-builder/1.0 (academic project)"}
INAT_FIELDS = ["role", "label", "taxon_query", "taxon_id", "obs_id", "obs_taxon", "user_id", "quality_grade",
               "captive", "observed_on", "photo_id", "photo_license", "attribution", "photo_url", "obs_url"]
import urllib.parse, urllib.request

def inat_api(path, **params):
    url = API + path + "?" + urllib.parse.urlencode(params, doseq=True)
    for attempt in range(5):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=60) as r:
                data = json.load(r)
            time.sleep(1.05)                      # iNaturalist asks for at most ~1 request per second
            return data
        except Exception as e:
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"iNaturalist API failed: {url}")

def resolve_taxon(name):
    for t in inat_api("taxa", q=name, per_page=10, is_active="true")["results"]:
        if t["name"].lower() == name.lower():
            return t["id"]
    raise ValueError(f"taxon not found exactly: {name}")

def collect_observations(role, label, taxon_query, taxon_id, want, user_cap, licenses, without_ids=()):
    rows, per_user, id_below, pages = [], {}, None, 0
    while len(rows) < want and pages < 40:
        params = dict(taxon_id=taxon_id, photos="true", photo_license=",".join(licenses), verifiable="any",
                      identifications="some_agree", per_page=200, order_by="id", order="desc")
        if without_ids:
            params["without_taxon_id"] = ",".join(map(str, without_ids))
        if id_below:
            params["id_below"] = id_below
        res = inat_api("observations", **params)["results"]; pages += 1
        if not res:
            break
        for o in res:
            id_below = o["id"]
            if o.get("num_identification_disagreements", 0) > 0 or not o.get("photos"):
                continue
            p = o["photos"][0]
            if p.get("license_code") not in licenses or not p.get("url"):
                continue
            uid = o["user"]["id"]
            if per_user.get(uid, 0) >= user_cap:
                continue
            per_user[uid] = per_user.get(uid, 0) + 1
            rows.append(dict(role=role, label=label, taxon_query=taxon_query, taxon_id=taxon_id, obs_id=o["id"],
                             obs_taxon=o["taxon"]["name"], user_id=uid, quality_grade=o["quality_grade"],
                             captive=o.get("captive"), observed_on=o.get("observed_on"), photo_id=p["id"],
                             photo_license=p["license_code"], attribution=p.get("attribution", ""),
                             photo_url=p["url"].replace("/square.", "/medium."),
                             obs_url=f"https://www.inaturalist.org/observations/{o['id']}"))
            if len(rows) >= want:
                break
    print(f"  {role:8s} {label:18s} {taxon_query:30s} -> {len(rows)}")
    return rows

def query_inat(cfg, out_dir):
    t, lic = cfg["targets"], cfg["inat_licenses"]
    rows = []
    for c in cfg["classes"]:
        want = int((t["class_total"] - (t["class_p100_max"] if c.get("p100") else 0)) * 1.25)  # +25% spare
        rows += collect_observations("class", c["slug"], c["taxon"], c["taxon_id"], want, t["inat_user_cap_class"], lic)
    for role, key, section in [("other", "other_inat_per_species", "other_plant"), ("openset", "openset_inat_per_species", "openset")]:
        for s in cfg[section]["inat"]:
            without = [resolve_taxon(w) for w in s.get("without", [])]
            rows += collect_observations(role, "other_plant" if role == "other" else "openset", s["taxon"],
                                         resolve_taxon(s["taxon"]), t[key], t["inat_user_cap_aux"], lic, without)
    seen, uniq = set(), []
    for r in rows:
        if r["obs_id"] not in seen:
            seen.add(r["obs_id"]); uniq.append(r)
    os.makedirs(out_dir, exist_ok=True)
    pd.DataFrame(uniq, columns=INAT_FIELDS).to_csv(os.path.join(out_dir, "inat_candidates.csv"), index=False)
    return len(uniq)

def download_file(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return True
    for attempt in range(4):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=60) as r:
                data = r.read()
            open(dest + ".part", "wb").write(data); os.replace(dest + ".part", dest)
            return True
        except Exception:
            time.sleep(3 * (attempt + 1))
    return False

def download_inat_photos(out_dir, workers=8):
    photos = os.path.join(out_dir, "photos"); os.makedirs(photos, exist_ok=True)
    rows = pd.read_csv(os.path.join(out_dir, "inat_candidates.csv"))
    with ThreadPoolExecutor(workers) as ex:
        ok = list(ex.map(lambda r: download_file(r[0], os.path.join(photos, f"{r[1]}.jpg")),
                         zip(rows.photo_url, rows.photo_id)))
    print(f"Downloaded {sum(ok)}/{len(ok)} iNaturalist photos")

PLANT100_URL = "https://huggingface.co/datasets/chandrabhuma/Plant100-CLS/resolve/main/Plant100_CLS.zip"
RAW_INAT = os.path.join(WORK_DIR, "raw", "inat")
PLANT100_ZIP = os.path.join(WORK_DIR, "raw", "plant100", "Plant100_CLS.zip")
P100_DIR = os.path.join(WORK_DIR, "interim", "p100")

if RUN_DATA_COLLECTION:
    print(query_inat(CONFIG, RAW_INAT), "iNaturalist candidates")
    download_inat_photos(RAW_INAT)
    os.makedirs(os.path.dirname(PLANT100_ZIP), exist_ok=True)
    subprocess.run(["curl", "-sSL", "--retry", "10", "-C", "-", "-o", PLANT100_ZIP, PLANT100_URL], check=True)
else:
    print("Skipping data collection (using the prepared dataset).")

## 4. Data cleaning and splitting

Steps, in order:
1. **Standardise every photo**: turn it the right way up (phone photos store their rotation separately, in
   "EXIF" data), convert to RGB, shrink so the longest side is 500 px, save as JPEG. Both sources go through
   the same steps, so the model cannot tell them apart by size or compression.
2. **Remove unreadable or tiny images** (shorter side under 160 px).
3. **Remove near-duplicates.** Each photo gets a **perceptual hash**, a 64-bit fingerprint that stays almost the
   same when a photo is resized or recompressed. Two photos whose fingerprints differ in 6 bits or fewer are
   treated as copies: one is kept. If copies appear under two *different* plant names, all of them are dropped.
4. **Fix labels.** Plant100's `Ficus_elastica` folder contains weeping figs (checked by eye), so it is used for
   `weeping_fig`; rubber plant photos come from iNaturalist only.
5. **Group related photos.** All photos by one iNaturalist photographer form one group. Each Plant100 folder is
   one group because its photos were taken in one or a few short sessions of the same plants.
6. **Split 70% / 15% / 15%** into training (to learn), validation (to make choices) and test (final exam).
   A group is never split across them. Plant100 bursts go to training only, so **validation and test are made of
   independent iNaturalist photos**. Plant100 is capped at 150 photos per class, picked evenly through the burst.
7. Open-set species go to a separate `openset` set that is never used for training.

In [ ]:
MAX_SIDE, JPEG_QUALITY, MIN_SIDE = 500, 90, 160
SPLITS = {"train": 0.70, "val": 0.15, "test": 0.15}

def normalize_image(src):
    # path or bytes -> RGB image, rotated upright, longest side <= 500 px
    im = Image.open(io.BytesIO(src) if isinstance(src, (bytes, bytearray)) else src)
    im.draft("RGB", (MAX_SIDE * 2, MAX_SIDE * 2))
    im = ImageOps.exif_transpose(im).convert("RGB")
    im.thumbnail((MAX_SIDE, MAX_SIDE), Image.Resampling.LANCZOS)
    return im

def capture_time(data, filename):
    try:
        exif = Image.open(io.BytesIO(data)).getexif()
        raw = exif.get_ifd(0x8769).get(36867) or exif.get(306)
        if raw:
            return pd.Timestamp(time.strftime("%Y-%m-%dT%H:%M:%S", time.strptime(str(raw).strip()[:19], "%Y:%m:%d %H:%M:%S")))
    except Exception:
        pass
    m = re.search(r"(20\d{6})[_-]?(\d{6})", filename)
    return pd.to_datetime(m.group(1) + m.group(2), format="%Y%m%d%H%M%S", errors="coerce") if m else None

def prepare_plant100(zip_path, cfg, out_dir, aux_sample=60, seed=SEED):
    # Read Plant100 straight from the zip: every image of the folders we use, a sample of the others.
    used = {f for c in cfg["classes"] for f in c.get("p100", [])}
    by_folder = defaultdict(list)
    z = zipfile.ZipFile(zip_path)
    for info in z.infolist():
        parts = info.filename.replace("\\", "/").split("/")
        if not info.is_dir() and len(parts) >= 2 and parts[-1].lower().endswith((".jpg", ".jpeg", ".png")):
            by_folder[parts[-2]].append(info.filename)
    rng, jobs = random.Random(seed), []
    for folder in sorted(by_folder):
        members = sorted(by_folder[folder])
        if folder not in used:
            members = rng.sample(members, min(aux_sample, len(members)))
        os.makedirs(os.path.join(out_dir, folder), exist_ok=True)
        jobs += [(m, folder, i) for i, m in enumerate(members)]
    def work(job):
        member, folder, idx = job
        data = z.read(member)
        ts, im = capture_time(data, os.path.basename(member)), normalize_image(data)
        if min(im.size) < MIN_SIDE:
            return None
        rel = f"{folder}/{idx:05d}.jpg"
        im.save(os.path.join(out_dir, rel), "JPEG", quality=JPEG_QUALITY, optimize=True)
        return dict(folder=folder, orig_name=os.path.basename(member), path=rel, width=im.size[0],
                    height=im.size[1], captured=ts.isoformat() if ts is not None and not pd.isna(ts) else "")
    with ThreadPoolExecutor(4) as ex:      # one zip handle: reading is serialised, decoding runs in parallel
        rows = [r for r in ex.map(work, jobs) if r]
    pd.DataFrame(rows).to_csv(os.path.join(out_dir, "p100_index.csv"), index=False)
    print(f"Plant100: {len(rows)} images prepared")

def phash_of(path, needs_norm):
    try:
        im = normalize_image(path) if needs_norm else Image.open(path).convert("RGB")
        return None if min(im.size) < MIN_SIDE else int(str(imagehash.phash(im)), 16)
    except Exception:
        return None

def duplicate_clusters(hashes, thresh):
    # Union-find: photos whose fingerprints differ by <= thresh bits end up in the same cluster.
    parent = list(range(len(hashes)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    h = np.array(hashes, dtype=np.uint64)
    for i in range(len(h) - 1):
        for j in np.nonzero(np.bitwise_count(h[i + 1:] ^ h[i]) <= thresh)[0]:
            a, b = find(i), find(i + 1 + int(j))
            if a != b:
                parent[b] = a
    return [find(i) for i in range(len(h))]

def spread_pick(recs, k):
    # k Plant100 frames evenly spaced through each burst (split evenly across folders)
    by_folder = defaultdict(list)
    for r in recs:
        by_folder[r["taxon"]].append(r)
    out = []
    for n, fo in enumerate(sorted(by_folder)):
        quota = k // len(by_folder) + (1 if n < k % len(by_folder) else 0)
        rs = sorted(by_folder[fo], key=lambda r: (r["captured"] or "9999", r["source_id"]))
        if quota >= len(rs):
            out += rs
        elif quota > 0:
            out += [rs[i] for i in sorted(set(np.linspace(0, len(rs) - 1, quota).round().astype(int)))]
    return out

def round_robin(recs, k, rng):
    # up to k photos, taking one photographer at a time (maximum variety)
    by_g = defaultdict(list)
    for r in recs:
        by_g[r["group"]].append(r)
    groups = list(by_g.values()); rng.shuffle(groups)
    for g in groups:
        rng.shuffle(g)
    out, i = [], 0
    while len(out) < k and groups:
        g = groups[i % len(groups)]
        out.append(g.pop())
        if not g:
            groups.remove(g)
        else:
            i += 1
    return out

def assign_splits(recs, fixed, rng):
    # Give each group (photographer) to train/val/test so every label stays close to 70/15/15.
    groups = defaultdict(list)
    for r in recs:
        groups[r["group"]].append(r)
    totals = Counter(r["label"] for r in recs + fixed)
    target = {l: {s: totals[l] * f for s, f in SPLITS.items()} for l in totals}
    cur = {l: Counter() for l in totals}
    for r in fixed:
        cur[r["label"]][r["split"]] += 1
    order = list(groups); rng.shuffle(order); order.sort(key=lambda g: -len(groups[g]))
    for g in order:
        cnt = Counter(r["label"] for r in groups[g])
        best = max(SPLITS, key=lambda s: sum(c * (target[l][s] - cur[l][s]) / max(target[l][s], 1) for l, c in cnt.items()))
        for r in groups[g]:
            r["split"] = best
        for l, c in cnt.items():
            cur[l][best] += c

def build_dataset(cfg, inat_dir, p100_dir, out, phash_thresh=6, seed=SEED):
    T, rng = cfg["targets"], random.Random(seed)
    meta = {c["slug"]: c for c in cfg["classes"]}
    folder_to_class = {f: c["slug"] for c in cfg["classes"] for f in c.get("p100", [])}
    openset_folders = set(cfg["openset"]["p100"])
    recs = []
    for r in csv.DictReader(open(os.path.join(inat_dir, "inat_candidates.csv"), encoding="utf-8")):
        p = os.path.join(inat_dir, "photos", f"{r['photo_id']}.jpg")
        if os.path.exists(p):
            recs.append(dict(label=r["label"], role=r["role"], source="inat", source_id=r["photo_id"],
                             group=f"inat_u{r['user_id']}", license=r["photo_license"], attribution=r["attribution"],
                             source_url=r["obs_url"], taxon=r["obs_taxon"], species=r["taxon_query"],
                             captured=r["observed_on"], raw=p, norm=True))
    for r in csv.DictReader(open(os.path.join(p100_dir, "p100_index.csv"), encoding="utf-8")):
        if not r["path"]:
            continue
        fo = r["folder"]
        label, role = ((folder_to_class[fo], "class") if fo in folder_to_class else
                       ("openset", "openset") if fo in openset_folders else ("other_plant", "other"))
        recs.append(dict(label=label, role=role, source="p100", source_id=r["path"].replace("/", "_")[:-4],
                         group=f"p100_{fo}", license="apache-2.0",
                         attribution="Plant100-CLS contributors (Bapatla Engineering College)",
                         source_url="https://huggingface.co/datasets/chandrabhuma/Plant100-CLS", taxon=fo,
                         species=fo, captured=r["captured"], raw=os.path.join(p100_dir, r["path"]), norm=False))
    print("Candidate images:", len(recs), dict(Counter(r["source"] for r in recs)))

    with ThreadPoolExecutor(8) as ex:
        hs = list(ex.map(lambda r: phash_of(r["raw"], r["norm"]), recs))
    bad = sum(h is None for h in hs)
    recs = [dict(r, phash=h) for r, h in zip(recs, hs) if h is not None]
    clusters = defaultdict(list)
    for r, root in zip(recs, duplicate_clusters([r["phash"] for r in recs], phash_thresh)):
        clusters[root].append(r)
    kept, dups, conflicts = [], 0, 0
    for members in clusters.values():
        if len({m["label"] for m in members}) > 1:
            conflicts += len(members); continue
        members.sort(key=lambda m: (m["source"] != "inat", m["source_id"]))
        kept.append(members[0]); dups += len(members) - 1
    print(f"Unreadable/tiny: {bad} | near-duplicates removed: {dups} | cross-label duplicates dropped: {conflicts}")

    pools = defaultdict(lambda: defaultdict(list))
    for r in kept:
        pools[r["label"]][r["source"]].append(r)
    fixed, flexible = [], []
    for slug in meta:
        p = spread_pick(pools[slug]["p100"], T["class_p100_max"])
        fixed += p; flexible += round_robin(pools[slug]["inat"], T["class_total"] - len(p), rng)
    by_folder = defaultdict(list)
    for r in pools["other_plant"]["p100"]:
        by_folder[r["taxon"]].append(r)
    fixed += [x for fo in sorted(by_folder) for x in spread_pick(by_folder[fo], T["other_p100_per_class"])]
    other_i = pools["other_plant"]["inat"]
    flexible += rng.sample(other_i, min(T["class_total"], len(other_i)))
    by_folder = defaultdict(list)
    for r in pools["openset"]["p100"]:
        by_folder[r["taxon"]].append(r)
    openset = [x for fo in sorted(by_folder) for x in spread_pick(by_folder[fo], T["openset_p100_per_class"])]
    openset += pools["openset"]["inat"]
    for r in openset:
        r["split"] = "openset"
    for r in fixed:
        r["split"] = "train"
    assign_splits(flexible, fixed, rng)

    labels = sorted(list(meta) + ["other_plant"]); idx = {l: i for i, l in enumerate(labels)}
    os.makedirs(out)
    rows = []
    for r in fixed + flexible + openset:
        rel = f"images/{r['split']}/{r['label']}/{r['source']}_{r['source_id']}.jpg"
        dest = os.path.join(out, rel); os.makedirs(os.path.dirname(dest), exist_ok=True)
        if r["norm"]:
            normalize_image(r["raw"]).save(dest, "JPEG", quality=JPEG_QUALITY, optimize=True)
        else:
            shutil.copyfile(r["raw"], dest)
        rows.append(dict(filepath=rel, split=r["split"], label=r["label"], label_index=idx.get(r["label"], -1),
                         role=r["role"], source=r["source"], source_id=r["source_id"], group=r["group"],
                         taxon=r["taxon"], species=r["species"], captured=r["captured"], license=r["license"],
                         attribution=r["attribution"], source_url=r["source_url"], phash=f"{r['phash']:016x}"))
    df = pd.DataFrame(rows).sort_values("filepath")
    df.to_csv(os.path.join(out, "metadata.csv"), index=False)
    open(os.path.join(out, "labels.txt"), "w").write("\n".join(labels) + "\n")
    classes = [dict(index=idx[l], slug=l, common=meta[l]["common"] if l in meta else "Other plant (not supported)",
                    scientific=meta[l]["scientific"] if l in meta else "-",
                    category=meta[l]["category"] if l in meta else "other") for l in labels]
    json.dump(classes, open(os.path.join(out, "classes.json"), "w", encoding="utf-8"), indent=1, ensure_ascii=False)
    json.dump(dict(near_dups_removed=dups, cross_label_dups_dropped=conflicts, unreadable_rejected=bad,
                   candidates=len(hs), total=len(df)), open(os.path.join(out, "build_summary.json"), "w"), indent=1)
    print("Dataset written:", out, len(df), "images")

def find_prepared_dataset():
    env = os.environ.get("PLANTID_DATA")
    cands = [env] if env else []
    if IN_KAGGLE:
        for root, dirs, files in os.walk("/kaggle/input"):
            if "labels.txt" in files and "metadata.csv" in files:
                cands.append(root); break
            dirs[:] = [d for d in dirs if d != "images"] if root.count("/") < 7 else []
    if IN_COLAB:
        dest = "/content/plantid_data"
        if not glob.glob(dest + "/**/labels.txt", recursive=True):
            zipfile.ZipFile("/content/drive/MyDrive/PlantID/plantid_v1.zip").extractall(dest)
        cands += [os.path.dirname(p) for p in glob.glob(dest + "/**/labels.txt", recursive=True)]
    cands += [os.path.join(WORK_DIR, "plantid_v1"), "./plantid_v1"]
    for c in cands:
        if c and os.path.exists(os.path.join(c, "labels.txt")):
            return c
    raise FileNotFoundError("Prepared dataset not found: add the plantid-v1 dataset or set PLANTID_DATA.")

if RUN_DATASET_BUILD:
    if RUN_DATA_COLLECTION or not os.path.exists(os.path.join(P100_DIR, "p100_index.csv")):
        prepare_plant100(PLANT100_ZIP, CONFIG, P100_DIR)
    DATA_ROOT = os.environ.get("PLANTID_BUILD_OUT", os.path.join(WORK_DIR, "plantid_v1_rebuilt"))
    build_dataset(CONFIG, RAW_INAT, P100_DIR, DATA_ROOT)
else:
    DATA_ROOT = find_prepared_dataset()
print("Dataset:", DATA_ROOT)

### Load the final dataset and check that nothing leaks between the splits

In [ ]:
meta = pd.read_csv(os.path.join(DATA_ROOT, "metadata.csv"), keep_default_na=False)
LABELS = [l.strip() for l in open(os.path.join(DATA_ROOT, "labels.txt"), encoding="utf-8") if l.strip()]
CLASSES = {c["slug"]: c for c in json.load(open(os.path.join(DATA_ROOT, "classes.json"), encoding="utf-8"))}
NUM_CLASSES, OTHER = len(LABELS), LABELS.index("other_plant")
meta["path"] = meta["filepath"].map(lambda p: os.path.join(DATA_ROOT, p))

assert all(os.path.exists(p) for p in meta["path"]), "some image files are missing"
g = meta[meta.split != "openset"].groupby("group")["split"].nunique()
assert (g == 1).all(), "a photographer/burst appears in more than one split"
assert set(meta[meta.split.isin(["val", "test"])].source) == {"inat"}, "val/test must be independent iNaturalist photos"
for col in ["species", "taxon"]:
    assert not set(meta[meta.split == "openset"][col]) & set(meta[meta.split != "openset"][col]), col
print(f"{len(meta):,} images | {NUM_CLASSES} classes | {meta.group.nunique():,} groups | all leakage checks passed")

train_df = meta[meta.split == "train"].sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = meta[meta.split == "val"].reset_index(drop=True)
test_df = meta[meta.split == "test"].reset_index(drop=True)
open_df = meta[meta.split == "openset"].reset_index(drop=True)
if QUICK_RUN:
    pick = lambda d, col, n: d.groupby(col).head(n).reset_index(drop=True)
    train_df, val_df, test_df = pick(train_df, "label", 8), pick(val_df, "label", 3), pick(test_df, "label", 3)
    open_df = pick(open_df, "species", 2)
meta.pivot_table(index="label", columns="split", values="filepath", aggfunc="count", fill_value=0)

## 5. Exploratory data analysis

In [ ]:
known = meta[meta.split != "openset"]
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
counts = known.groupby(["label", "source"]).size().unstack(fill_value=0).loc[LABELS]
counts.plot(kind="barh", stacked=True, ax=ax[0], color={"inat": "#4c9a6a", "p100": "#d9a441"})
ax[0].set_title("Photos per class and source"); ax[0].set_xlabel("photos"); ax[0].invert_yaxis()
lic = meta.license.value_counts()
ax[1].bar(lic.index, lic.values, color="#4c9a6a"); ax[1].set_title("Licences"); ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "figures", "eda_counts.png"), dpi=110); plt.show()
print("Share of Plant100 photos per class (training only):")
print((known.groupby("label").source.apply(lambda s: (s == "p100").mean() * 100)).round(0).sort_values(ascending=False).head(15).to_dict())

In [ ]:
# One random example per class
fig, axes = plt.subplots(4, 8, figsize=(18, 10))
for ax, label in zip(axes.flat, LABELS + [None]):
    ax.axis("off")
    if label:
        r = known[known.label == label].sample(1, random_state=1).iloc[0]
        ax.imshow(Image.open(r.path)); ax.set_title(f"{CLASSES[label]['common'][:22]}\n[{r.source}]", fontsize=8)
plt.suptitle("One example per class"); plt.tight_layout(); plt.show()

# Image sizes after standardising
sizes = np.array([Image.open(p).size for p in known.sample(min(800, len(known)), random_state=0).path])
print("Width x height (sample of 800): median", np.median(sizes, 0), "| min", sizes.min(0), "| max", sizes.max(0))
print("Portrait photos:", f"{(sizes[:, 1] > sizes[:, 0]).mean():.0%}")

In [ ]:
# Plant100 capture times: how long did it take to photograph each class?
p = known[(known.source == "p100") & (known.captured != "")].copy()
p["t"] = pd.to_datetime(p["captured"], errors="coerce")
span = p.groupby("species")["t"].agg(start="min", end="max", photos="count")
span["minutes"] = (span.end - span.start).dt.total_seconds() / 60
span["days"] = p.groupby("species")["t"].apply(lambda s: s.dt.date.nunique())
display(span.sort_values("minutes")[["start", "minutes", "days", "photos"]].round(1))
print(f"{(span.days == 1).sum()} of {len(span)} Plant100 folders were photographed on a single day; "
      f"median session length {span[span.days == 1].minutes.median():.0f} minutes")
fig, ax = plt.subplots(figsize=(9, 3.5))
for sp in list(span.sort_values("photos", ascending=False).index[:6]):
    t = p[p.species == sp]["t"].sort_values()
    ax.plot((t - t.iloc[0]).dt.total_seconds() / 60, range(len(t)), label=sp)
ax.set_xlabel("minutes since first photo of the class"); ax.set_ylabel("photo number"); ax.legend(fontsize=7)
ax.set_title("Plant100: most classes were photographed in one short session")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "figures", "eda_plant100_bursts.png"), dpi=110); plt.show()

### Observations

1. **Plant100 classes are short photo sessions.** Most Plant100 folders were photographed **on one day within
   about 5–45 minutes**, with photos only seconds apart; a few (Hibiscus, Bougainvillea) span 2–4 days (table and
   chart above). Most photos in a folder are therefore near-copies of the same few plants. This explains the base
   model's 100% score and is why Plant100 is used for training only and capped at 150 photos per class.
2. **One Plant100 label was wrong.** The `Ficus_elastica` folder shows weeping figs (small pointed leaves on
   drooping stems), and the `Ficus_benjamina` folder only the striped (variegated) form. The base model was
   partly learning "striped vs. green" rather than two species. After the fix, `weeping_fig` has both forms.
3. **The classes are balanced, except one.** Every species has 900 photos (630 train / 135 val / 135 test) except
   **Ming aralia (535)**, because iNaturalist has few verified photos of it; class weights compensate during training.
   `other_plant` is larger (1,225) on purpose because it has to cover many different plants.
4. **The photos are varied.** About 91% of photos come from iNaturalist, taken by thousands of different people
   indoors and outdoors with different phones. Validation and test contain only these independent photos.
5. **Duplicates were rare but real.** Cleaning removed 619 near-duplicate photos, 3 photos filed under two
   different plants, and 7 unreadable files (from 46,215 candidates).
6. **Licences.** About 87% of photos are CC BY-NC (non-commercial use only). This is fine for a university project;
   a commercial release would need these photos replaced.

## 6. Preparing images for the model

**At prediction time (the app must do exactly this):** turn the photo upright → crop the central square → resize
to 224 × 224 pixels → pass the RGB values (0–255) to the model. The model normalises the values itself.

**Data augmentation, training only.** During training each photo is changed randomly every time it is shown:
random crop (30–100% of the photo), mirror, small rotation, brightness/contrast/colour changes, occasional blur
and JPEG compression. This imitates different phones, distances and lighting, so the model learns the plant
rather than one particular photo. Validation and test photos are never augmented.

In [ ]:
IMG_SIZE = 224
BATCH = 16 if QUICK_RUN else 64
AUTOTUNE = tf.data.AUTOTUNE
_rotate = layers.RandomRotation(0.08, fill_mode="reflect", seed=SEED)

def decode(img_bytes):
    return tf.cast(tf.io.decode_jpeg(img_bytes, channels=3), tf.float32)

def eval_preprocess(img_bytes):
    img = decode(img_bytes)
    h, w = tf.shape(img)[0], tf.shape(img)[1]; s = tf.minimum(h, w)
    img = tf.image.crop_to_bounding_box(img, (h - s) // 2, (w - s) // 2, s, s)
    return tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method="bilinear", antialias=True)

def train_preprocess(img_bytes):
    img = decode(img_bytes)
    begin, size, _ = tf.image.sample_distorted_bounding_box(
        tf.shape(img), bounding_boxes=tf.constant([[[0.0, 0.0, 1.0, 1.0]]]), min_object_covered=0.0,
        aspect_ratio_range=(3 / 4, 4 / 3), area_range=(0.30, 1.0), max_attempts=20, use_image_if_no_bounding_boxes=True)
    img = tf.image.resize(tf.slice(img, begin, size), [IMG_SIZE, IMG_SIZE], antialias=True)
    img = _rotate(tf.image.random_flip_left_right(img), training=True)
    x = img / 255.0
    x = tf.image.random_brightness(x, 0.15)
    x = tf.image.random_contrast(x, 0.8, 1.2)
    x = tf.image.random_saturation(x, 0.75, 1.25)
    x = tf.clip_by_value(tf.image.random_hue(x, 0.03), 0.0, 1.0)
    if tf.random.uniform([]) < 0.2:
        x = tf.image.resize(tf.image.resize(x, [IMG_SIZE // 2, IMG_SIZE // 2]), [IMG_SIZE, IMG_SIZE])
    if tf.random.uniform([]) < 0.3:
        x = tf.reshape(tf.image.random_jpeg_quality(x, 40, 95), [IMG_SIZE, IMG_SIZE, 3])
    return tf.clip_by_value(x, 0.0, 1.0) * 255.0

def baseline_preprocess(img_bytes):            # the base notebook simply resized the whole photo
    return tf.image.resize(decode(img_bytes), [IMG_SIZE, IMG_SIZE])

def make_ds(df, preprocess, training=False, batch=None):
    y = np.where(df["label_index"].values >= 0, df["label_index"].values, 0)
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, y))
    ds = ds.map(lambda p, l: (tf.io.read_file(p), l), num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(min(len(df), 8192), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda b, l: (preprocess(b), tf.one_hot(l, NUM_CLASSES)), num_parallel_calls=AUTOTUNE)
    return ds.batch(batch or BATCH).prefetch(AUTOTUNE)

labels_of = lambda df: df["label_index"].values.astype(int)
y_val, y_test = labels_of(val_df), labels_of(test_df)
train_ds, val_ds, test_ds = make_ds(train_df, train_preprocess, True), make_ds(val_df, eval_preprocess), make_ds(test_df, eval_preprocess)
open_ds = make_ds(open_df, eval_preprocess)

ex = next(iter(make_ds(train_df.head(8), train_preprocess, True, 8)))[0].numpy()
fig, ax = plt.subplots(1, 8, figsize=(16, 2.4))
for i in range(8):
    ax[i].imshow(ex[i].astype("uint8")); ax[i].axis("off")
plt.suptitle("Augmented training images"); plt.show()

counts_tr = train_df["label_index"].value_counts().reindex(range(NUM_CLASSES), fill_value=0)
CLASS_WEIGHT = {i: float(len(train_df) / (NUM_CLASSES * max(c, 1))) for i, c in counts_tr.items()}   # rarer class -> larger weight

### Shared helpers: scoring and training

**Metrics used.**
* **Accuracy**: share of photos named correctly.
* **Precision** for a plant: when the model says "basil", how often is it really basil?
* **Recall**: of all real basil photos, how many did the model find?
* **F1**: one number balancing precision and recall. **Macro-F1** averages F1 over all classes equally, so a weak
  class cannot hide behind strong ones. This is our main score.
* **Top-3 accuracy**: the correct plant is among the model's three best guesses (useful for an app that shows
  three suggestions).

In [ ]:
def metrics(y, probs):
    pred = probs.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=range(NUM_CLASSES), zero_division=0)
    return dict(accuracy=float((pred == y).mean()), macro_precision=float(p.mean()), macro_recall=float(r.mean()),
                macro_f1=float(f.mean()), top3=float(top_k_accuracy_score(y, probs, k=3, labels=range(NUM_CLASSES))))

def merge_hist(*hs):
    out = {}
    for h in hs:
        for k, v in h.history.items():
            out.setdefault(k, []).extend(float(x) for x in v)
    out["phase1_epochs"] = len(hs[0].history["loss"])
    return out

def model_paths(name):
    return os.path.join(OUT_DIR, "models", f"{name}.keras"), os.path.join(OUT_DIR, "logs", f"{name}_history.json")

def build_model(arch, weights="imagenet", head_dropout=0.2):
    # Input: RGB 0-255. Each architecture's own normalisation is inside the model.
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    shape = (IMG_SIZE, IMG_SIZE, 3)
    if arch == "mobilenet_v2":
        base = keras.applications.MobileNetV2(input_shape=shape, include_top=False, weights=weights, pooling="avg")
        x = layers.Rescaling(1 / 127.5, offset=-1.0, name="normalize")(inp)
    elif arch == "mobilenet_v3_large":
        base = keras.applications.MobileNetV3Large(input_shape=shape, include_top=False, weights=weights,
                                                   pooling="avg", include_preprocessing=True)
        x = inp
    elif arch == "efficientnet_b0":
        base = keras.applications.EfficientNetB0(input_shape=shape, include_top=False, weights=weights, pooling="avg")
        x = inp
    x = base(x, training=False)
    x = layers.Dropout(head_dropout)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="probs")(x)
    return keras.Model(inp, out, name=f"plantid_{arch}"), base

def compile_model(model, lr, smoothing=0.1):
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=smoothing),
                  metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=3, name="top3")])

RESULTS, PROBS, HISTS = {}, {}, {}
def record(name, kind, model_or_params, pv, pt, hist=None, size_mb=None):
    params = model_or_params if isinstance(model_or_params, int) else int(model_or_params.count_params())
    RESULTS[name] = dict(kind=kind, params=params, size_mb=size_mb, val=metrics(y_val, pv), test=metrics(y_test, pt),
                         epochs=len(hist["loss"]) if hist else None, train_min=(hist or {}).get("train_seconds", 0) / 60)
    PROBS[name] = dict(val=pv, test=pt)
    if hist:
        HISTS[name] = hist
    print(f"{name}: val macro-F1 {RESULTS[name]['val']['macro_f1']:.3f} | test accuracy {RESULTS[name]['test']['accuracy']:.3f}")

## 7. Base model, retrained on 30 species

We first repeat the **exact recipe of `Model_Base.ipynb`** (MobileNetV2, light augmentation inside the model,
plain resize, 15 + 15 epochs) on the new data. Only the data has changed, so this shows how the base model
performs on the harder, honest test, and gives a reference point for the improvements in Section 8.

In [ ]:
def train_base_recipe():
    path, hpath = model_paths("base_mobilenet_v2")
    if os.path.exists(path):
        print("already trained, loading"); return keras.models.load_model(path), json.load(open(hpath))
    ep = 1 if QUICK_RUN else 15
    tr, va = make_ds(train_df, baseline_preprocess, True, 32), make_ds(val_df, baseline_preprocess, batch=32)
    augment = keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.1),
                                layers.RandomZoom(0.1), layers.RandomBrightness(0.1)])
    base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
    base.trainable = False
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1 / 127.5, offset=-1.0)(augment(inp))
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(base(x, training=False)))
    model = keras.Model(inp, layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x))
    cbs = lambda: [keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_accuracy"),
                   keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, monitor="val_loss")]
    t0 = time.time()
    model.compile(keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    h1 = model.fit(tr, validation_data=va, epochs=ep, callbacks=cbs(), verbose=2)
    base.trainable = True
    for l in base.layers[:-30]:
        l.trainable = False
    model.compile(keras.optimizers.Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"])
    h2 = model.fit(tr, validation_data=va, epochs=ep, callbacks=cbs(), verbose=2)
    hist = merge_hist(h1, h2); hist["train_seconds"] = time.time() - t0
    model.save(path); json.dump(hist, open(hpath, "w"))
    return model, hist

m, h = train_base_recipe()
record("base_recipe (MobileNetV2)", "base model, original recipe", m,
       m.predict(make_ds(val_df, baseline_preprocess, batch=32), verbose=0),
       m.predict(make_ds(test_df, baseline_preprocess, batch=32), verbose=0), h)
del m; keras.backend.clear_session(); gc.collect()

### Cross-validation of the base model

**Cross-validation** checks that a result is not luck from one particular split. The training + validation
photos are divided into 5 parts (**folds**); the model is trained on 4 parts and tested on the 5th, five times
over, and we report the average and the spread. Folds are made by **photographer group**, so the same person's
photos are never on both sides.

To keep this affordable we cross-validate the base model's first phase: the frozen MobileNetV2 turns every photo
into 1,280 numbers (**features**), and a softmax layer is trained on them. A softmax layer on fixed features is
the same thing as **multinomial logistic regression**, so we use scikit-learn's version. The same features also
give us one of the comparison models in Section 8.

In [ ]:
feat_model = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                                            weights="imagenet", pooling="avg")
def features(df):
    ds = make_ds(df, eval_preprocess).map(lambda x, y: x / 127.5 - 1.0)      # MobileNetV2's normalisation
    return feat_model.predict(ds, verbose=0).astype(np.float32)

cv_df = pd.concat([train_df, val_df], ignore_index=True)
F_cv, F_test = features(cv_df), features(test_df)
y_cv, g_cv = labels_of(cv_df), cv_df["group"].values
n_folds = 2 if QUICK_RUN else 5
fold_scores = []
for k, (tr_i, va_i) in enumerate(StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=SEED).split(F_cv, y_cv, g_cv)):
    sc = StandardScaler().fit(F_cv[tr_i])
    clf = LogisticRegression(max_iter=300, C=0.5).fit(sc.transform(F_cv[tr_i]), y_cv[tr_i])
    pr = clf.predict_proba(sc.transform(F_cv[va_i]))
    probs = np.zeros((len(va_i), NUM_CLASSES)); probs[:, clf.classes_] = pr
    fold_scores.append(metrics(y_cv[va_i], probs))
    print(f"fold {k + 1}: accuracy {fold_scores[-1]['accuracy']:.3f} | macro-F1 {fold_scores[-1]['macro_f1']:.3f}")
CV = {m: (float(np.mean([f[m] for f in fold_scores])), float(np.std([f[m] for f in fold_scores]))) for m in fold_scores[0]}
print(f"\nCross-validation ({n_folds} folds): accuracy {CV['accuracy'][0]:.3f} ± {CV['accuracy'][1]:.3f} | "
      f"macro-F1 {CV['macro_f1'][0]:.3f} ± {CV['macro_f1'][1]:.3f}")

# Comparison model: logistic regression on frozen MobileNetV2 features (trained on train only, scored on val/test)
F_tr, F_val = F_cv[:len(train_df)], F_cv[len(train_df):]
sc = StandardScaler().fit(F_tr)
lr_clf = LogisticRegression(max_iter=300, C=0.5).fit(sc.transform(F_tr), labels_of(train_df))
def lr_probs(F):
    out = np.zeros((len(F), NUM_CLASSES)); out[:, lr_clf.classes_] = lr_clf.predict_proba(sc.transform(F)); return out
record("logistic regression on MobileNetV2 features", "linear classifier on frozen features",
       int(feat_model.count_params() + lr_clf.coef_.size + lr_clf.intercept_.size), lr_probs(F_val), lr_probs(F_test))
del feat_model; keras.backend.clear_session(); gc.collect()

## 8. Comparison models

All models below use the same data, the same split and the same scoring.

| Model | What it is | Why it is here |
|---|---|---|
| Small CNN from scratch | 4 convolution blocks, no pretrained knowledge | Shows how much transfer learning helps |
| Logistic regression on features | Section 7: a linear classifier on frozen MobileNetV2 features | Simplest possible model on top of pretrained features |
| MobileNetV2 (improved recipe) | Base architecture with the improved training below | Same network as the base model, better training |
| MobileNetV3-Large | Newer phone network | Usually more accurate at similar speed |
| EfficientNet-B0 | Phone-sized network that is accurate but heavier | Accuracy vs. size trade-off |

**Improved training recipe:** stronger augmentation (Section 6); fine-tuning of the whole network instead of
only the last 30 layers; **label smoothing** (targets of 0.9 instead of 1.0, which stops the model from becoming
overconfident); **AdamW** with **weight decay** (keeps weights small, a form of regularisation); a learning rate
that warms up and then decreases smoothly (**cosine schedule**); class weights; and **early stopping** (stop when
validation loss stops improving and keep the best epoch).

In [ ]:
HEAD_EPOCHS, FT_EPOCHS, PATIENCE, HEAD_LR, FT_LR = (1, 1, 1, 1e-3, 1e-4) if QUICK_RUN else (5, 30, 6, 1e-3, 1e-4)

def train_transfer(arch):
    path, hpath = model_paths(arch)
    if os.path.exists(path):
        print(f"[{arch}] already trained, loading"); return keras.models.load_model(path), json.load(open(hpath))
    model, base = build_model(arch)
    t0 = time.time()
    base.trainable = False                                   # phase 1: train the new head only
    compile_model(model, HEAD_LR)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=CLASS_WEIGHT, verbose=2)
    base.trainable = True                                    # phase 2: fine-tune everything except BatchNorm
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False
    steps_per_epoch = math.ceil(len(train_df) / BATCH)
    sched = keras.optimizers.schedules.CosineDecay(FT_LR / 10, decay_steps=max(steps_per_epoch * (FT_EPOCHS - 1), 1),
                                                   alpha=0.01, warmup_target=FT_LR, warmup_steps=steps_per_epoch)
    compile_model(model, sched)
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
           keras.callbacks.ModelCheckpoint(path, monitor="val_loss", save_best_only=True)]
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=FT_EPOCHS, class_weight=CLASS_WEIGHT, callbacks=cbs, verbose=2)
    hist = merge_hist(h1, h2); hist["train_seconds"] = time.time() - t0
    model.save(path); json.dump(hist, open(hpath, "w"))
    return model, hist

def train_small_cnn():
    path, hpath = model_paths("small_cnn")
    if os.path.exists(path):
        print("[small_cnn] already trained, loading"); return keras.models.load_model(path), json.load(open(hpath))
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1 / 255.0)(inp)
    for f in [32, 64, 128, 256]:
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.Activation("relu")(layers.BatchNormalization()(x))
        x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    model = keras.Model(inp, layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x), name="small_cnn")
    compile_model(model, 1e-3)
    t0 = time.time()
    h = model.fit(train_ds, validation_data=val_ds, epochs=1 if QUICK_RUN else 30, class_weight=CLASS_WEIGHT, verbose=2,
                  callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)])
    hist = merge_hist(h); hist["train_seconds"] = time.time() - t0
    model.save(path); json.dump(hist, open(hpath, "w"))
    return model, hist

m, h = train_small_cnn()
record("small CNN from scratch", "CNN without transfer learning", m, m.predict(val_ds, verbose=0), m.predict(test_ds, verbose=0), h)
del m; keras.backend.clear_session(); gc.collect()

ARCHS = ["mobilenet_v2", "mobilenet_v3_large", "efficientnet_b0"]
for arch in ARCHS:
    print(f"\n===== {arch} =====")
    m, h = train_transfer(arch)
    record(arch, "transfer learning, improved recipe", m, m.predict(val_ds, verbose=0), m.predict(test_ds, verbose=0), h,
           size_mb=os.path.getsize(model_paths(arch)[0]) / 1e6)
    del m; keras.backend.clear_session(); gc.collect()

In [ ]:
comparison = pd.DataFrame([dict(model=n, kind=r["kind"], params_M=r["params"] / 1e6, val_macro_f1=r["val"]["macro_f1"],
                                test_accuracy=r["test"]["accuracy"], test_macro_f1=r["test"]["macro_f1"],
                                test_top3=r["test"]["top3"], epochs=r["epochs"], train_min=r["train_min"])
                           for n, r in RESULTS.items()]).set_index("model").round(3)
comparison.to_csv(os.path.join(OUT_DIR, "logs", "model_comparison.csv"))
display(comparison)
print(f"Base model cross-validation: macro-F1 {CV['macro_f1'][0]:.3f} ± {CV['macro_f1'][1]:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
c = comparison.sort_values("test_macro_f1")
ax[0].barh(c.index, c.test_macro_f1, color="#4c9a6a"); ax[0].set_xlabel("test macro-F1"); ax[0].set_xlim(0, 1)
ax[0].set_title("Model comparison (higher is better)")
for i, v in enumerate(c.test_macro_f1):
    ax[0].text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=8)
ax[1].scatter(comparison.params_M, comparison.test_macro_f1, color="#d9a441")
for n, r in comparison.iterrows():
    ax[1].annotate(n.split(" (")[0][:24], (r.params_M, r.test_macro_f1), fontsize=7)
ax[1].set_xlabel("parameters (millions) — smaller fits phones better"); ax[1].set_ylabel("test macro-F1")
ax[1].set_title("Accuracy vs. size")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "figures", "model_comparison.png"), dpi=120); plt.show()

## 9. Choosing and evaluating the final model

**Choice rule (validation data only; the test set plays no part):** among the three phone networks, take the best
validation macro-F1, unless a smaller network is within 1 percentage point of it, in which case take the smaller one.

In [ ]:
best = max(RESULTS[a]["val"]["macro_f1"] for a in ARCHS)
FINAL_ARCH = min([a for a in ARCHS if RESULTS[a]["val"]["macro_f1"] >= best - 0.01], key=lambda a: RESULTS[a]["params"])
print("Validation macro-F1:", {a: round(RESULTS[a]["val"]["macro_f1"], 4) for a in ARCHS}, "\nFINAL MODEL:", FINAL_ARCH)
final_model = keras.models.load_model(model_paths(FINAL_ARCH)[0])
p_val, p_test = PROBS[FINAL_ARCH]["val"], PROBS[FINAL_ARCH]["test"]
FINAL_TEST = RESULTS[FINAL_ARCH]["test"]

h = HISTS[FINAL_ARCH]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, keys, t in [(ax[0], ["accuracy", "val_accuracy"], "Accuracy"), (ax[1], ["loss", "val_loss"], "Loss")]:
    for k in keys:
        a.plot(h[k], label=k)
    a.axvline(h["phase1_epochs"] - 0.5, color="grey", ls="--", lw=1); a.set_title(f"{FINAL_ARCH}: {t}"); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "figures", "training_curves.png"), dpi=120); plt.show()

test_pred = p_test.argmax(1)
print(classification_report(y_test, test_pred, labels=range(NUM_CLASSES), target_names=LABELS, zero_division=0, digits=3))
REPORT = classification_report(y_test, test_pred, labels=range(NUM_CLASSES), target_names=LABELS, zero_division=0, output_dict=True)

tr_sample = train_df.sample(min(3000, len(train_df)), random_state=SEED)
FIT = dict(train=metrics(labels_of(tr_sample), final_model.predict(make_ds(tr_sample, eval_preprocess), verbose=0))["accuracy"],
           val=RESULTS[FINAL_ARCH]["val"]["accuracy"], test=FINAL_TEST["accuracy"])
print("Overfitting check (accuracy)  train {train:.3f} | val {val:.3f} | test {test:.3f}".format(**FIT))
print("A large gap between train and test would mean memorising; a small gap means the model generalises.")

In [ ]:
cm = confusion_matrix(y_test, test_pred, labels=range(NUM_CLASSES))
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cmn, cmap="Greens", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(LABELS, rotation=90, fontsize=8)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(LABELS, fontsize=8)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=6, color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Confusion matrix, test set ({FINAL_ARCH})")
plt.colorbar(im, fraction=0.03); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figures", "confusion_matrix.png"), dpi=130); plt.show()
CONFUSED = sorted([(LABELS[i], LABELS[j], int(cm[i, j])) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)
                   if i != j and cm[i, j]], key=lambda x: -x[2])[:12]
print("Most frequent mix-ups (true -> predicted):")
display(pd.DataFrame(CONFUSED, columns=["true", "predicted", "photos"]))

**Reading the confusion matrix:** each row is the real plant, each column the model's answer. A perfect model
has numbers only on the diagonal. Off-diagonal numbers show which plants get mixed up; these are usually plants
that look alike (for example plants without flowers, or seedlings).

## 10. Unknown plants and invalid photos

A normal classifier always picks one of its classes, even for a plant it has never seen. PlantID has two safeguards:
1. the **`other_plant` class**, trained on look-alikes and many other species;
2. a **confidence threshold**. The model's probabilities are first **calibrated** with a single number called the
   **temperature** (so that "80% sure" really means right about 80% of the time). If the top answer is below the
   threshold, the app says *"Not sure which plant this is"* and shows the top-3 guesses instead.

The threshold is tuned on validation photos and on half of the never-seen species; the other half of the never-seen
species is used only to report the result.

In [ ]:
def calibrate(probs, T):
    z = np.log(np.clip(probs, 1e-12, 1.0)) / T
    z -= z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

nll = lambda probs, y: float(-np.log(np.clip(probs[np.arange(len(y)), y], 1e-12, 1)).mean())
TEMPERATURE = float(min(np.round(np.linspace(0.5, 3.0, 51), 3), key=lambda t: nll(calibrate(p_val, t), y_val)))

p_open = final_model.predict(open_ds, verbose=0)
species = sorted(open_df["species"].unique()); tune_species = set(species[::2])
is_tune = open_df["species"].isin(tune_species).values

def decide(probs, tau):
    pred, conf = probs.argmax(1), probs.max(1)
    return pred, (pred != OTHER) & (conf >= tau)            # accepted = named a supported plant

def known_stats(probs, y, tau):
    m = y != OTHER; pred, acc = decide(probs[m], tau); ok = pred == y[m]
    return dict(answered=float(acc.mean()), accuracy_when_answered=float(ok[acc].mean()) if acc.any() else 0.0,
                correct_and_answered=float((ok & acc).mean()))

rejected = lambda probs, tau: float((~decide(probs, tau)[1]).mean())
cv_, co = calibrate(p_val, TEMPERATURE), calibrate(p_open[is_tune], TEMPERATURE)
curve = pd.DataFrame([dict(tau=t, **known_stats(cv_, y_val, t), unknown_rejected=rejected(co, t))
                      for t in np.round(np.arange(0.0, 0.96, 0.01), 2)])
curve["score"] = 0.5 * curve.correct_and_answered + 0.5 * curve.unknown_rejected
ok = curve[curve.answered >= 0.85]                               # still answer at least 85% of known plants
THRESHOLD = float((ok if len(ok) else curve).sort_values("score", ascending=False).iloc[0].tau)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve.tau, curve.answered, label="known plants answered")
ax.plot(curve.tau, curve.accuracy_when_answered, label="accuracy when answered")
ax.plot(curve.tau, curve.unknown_rejected, label="unknown plants rejected")
ax.axvline(THRESHOLD, color="grey", ls="--"); ax.set_xlabel("confidence threshold"); ax.legend(); ax.set_title("Choosing the threshold")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "figures", "threshold.png"), dpi=120); plt.show()

ct, ce = calibrate(p_test, TEMPERATURE), calibrate(p_open[~is_tune], TEMPERATURE)
UNKNOWN = dict(temperature=TEMPERATURE, threshold=THRESHOLD, test_known=known_stats(ct, y_test, THRESHOLD),
               test_other_plant_rejected=rejected(ct[y_test == OTHER], THRESHOLD) if (y_test == OTHER).any() else None,
               never_seen_rejected=rejected(ce, THRESHOLD), never_seen_rejected_without_threshold=rejected(ce, 0.0),
               never_seen_species=sorted(set(species) - tune_species))
print(json.dumps(UNKNOWN, indent=1))

## 11. Explainability: what the model looked at

**Grad-CAM** colours the parts of a photo that pushed the model towards its answer (red = strong influence).
It works by measuring how much each region of the last convolutional layer changes the winning class's score.
If the colour sits on leaves or flowers, the model is using the plant; if it sits on the pot or the background,
that prediction should not be trusted. The app can show this kind of explanation next to the top-3 answers.

In [ ]:
_base = next(l for l in final_model.layers if isinstance(l, keras.Model))
_pos = final_model.layers.index(_base)
_pre, _head = final_model.layers[1:_pos], final_model.layers[_pos + 1:]
_conv = [l for l in _base.layers if len(l.output.shape) == 4][-1]
_conv_model = keras.Model(_base.inputs, [_conv.output, _base.output])

def gradcam(img224, class_idx=None):
    x = tf.convert_to_tensor(img224[None], tf.float32)
    with tf.GradientTape() as tape:
        for l in _pre:
            x = l(x)
        conv_out, pooled = _conv_model(x, training=False)
        y = pooled
        for l in _head:
            y = l(y, training=False)
        y = tf.cast(y, tf.float32)
        class_idx = int(tf.argmax(y[0])) if class_idx is None else class_idx
        score = y[:, class_idx]
    grads = tf.cast(tape.gradient(score, conv_out), tf.float32)[0]
    weights = tf.reduce_mean(grads, axis=(0, 1))
    cam = tf.nn.relu(tf.reduce_sum(tf.cast(conv_out[0], tf.float32) * weights, axis=-1)).numpy()
    cam = cam / (cam.max() + 1e-8)
    return np.array(Image.fromarray(np.uint8(cam * 255)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0, class_idx, float(y[0, class_idx])

def load224(path):
    return eval_preprocess(tf.io.read_file(path)).numpy()

samples = test_df.groupby("label").head(1).sample(min(8, NUM_CLASSES), random_state=3)
fig, axes = plt.subplots(2, len(samples), figsize=(2.6 * len(samples), 6.4), layout="constrained")
for k, (_, r) in enumerate(samples.iterrows()):
    img = load224(r.path); cam, ci, conf = gradcam(img)
    axes[0, k].imshow(img.astype("uint8")); axes[0, k].set_title(f"true: {CLASSES[r.label]['common'][:16]}", fontsize=8)
    axes[1, k].imshow(img.astype("uint8")); axes[1, k].imshow(cam, cmap="jet", alpha=0.45)
    axes[1, k].set_title(f"pred: {CLASSES[LABELS[ci]]['common'][:16]} {conf:.0%}", fontsize=8)
    axes[0, k].axis("off"); axes[1, k].axis("off")
plt.suptitle("Grad-CAM: regions that influenced the prediction")
plt.savefig(os.path.join(OUT_DIR, "figures", "gradcam.png"), dpi=110); plt.show()

## 12. Export for Android (TensorFlow Lite) and verification

**TensorFlow Lite (LiteRT)** is the format Android apps use to run models on the phone. We make three versions:
* **float32**: exact copy, largest file;
* **float16**: numbers stored with half the precision, about half the size;
* **int8**: numbers stored as small integers, about a quarter of the size (**quantisation**), calibrated on 300
  training photos.

Every version keeps the same input (RGB 0–255, 224 × 224) and output (31 probabilities), so the app code does not
change. Each version is run on the **whole test set** and compared with the original model. The recommended file is
the smallest one that stays within 1 point of macro-F1 and agrees with the original on at least 98% of photos.
Finally, the predictions are repeated with a completely separate image pipeline (Pillow instead of TensorFlow), the way
an app developer would write it, to check that the documented preprocessing gives the same answers.

In [ ]:
keras.mixed_precision.set_global_policy("float32")
export_model, _ = build_model(FINAL_ARCH, weights=None)
export_model.set_weights(final_model.get_weights())
sm_dir = os.path.join(OUT_DIR, "export", "saved_model"); shutil.rmtree(sm_dir, ignore_errors=True)
export_model.export(sm_dir)

def rep_data():
    for p in train_df.sample(min(300, len(train_df)), random_state=SEED)["path"]:
        yield [tf.expand_dims(eval_preprocess(tf.io.read_file(p)), 0)]

def convert(kind):
    c = tf.lite.TFLiteConverter.from_saved_model(sm_dir)
    if kind == "float16":
        c.optimizations = [tf.lite.Optimize.DEFAULT]; c.target_spec.supported_types = [tf.float16]
    elif kind == "int8":
        c.optimizations = [tf.lite.Optimize.DEFAULT]; c.representative_dataset = rep_data
        c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    path = os.path.join(OUT_DIR, "export", f"plantid_{FINAL_ARCH}_{kind}.tflite")
    open(path, "wb").write(c.convert())
    return path

BATCH_TFL = 32
def tf_batches(df):        # photos prepared by the TensorFlow pipeline, 32 at a time
    return (x for x, _ in make_ds(df, eval_preprocess, batch=BATCH_TFL).as_numpy_iterator())

def tflite_predict(path, batches, batch=BATCH_TFL):
    it = TFLiteInterpreter(model_path=path, num_threads=4)
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    it.resize_tensor_input(i["index"], [batch, IMG_SIZE, IMG_SIZE, 3]); it.allocate_tensors()
    outs = []
    for x in batches:
        n = len(x)
        if n < batch:
            x = np.concatenate([x, np.zeros((batch - n, IMG_SIZE, IMG_SIZE, 3), np.float32)])
        it.set_tensor(i["index"], x.astype(np.float32)); it.invoke(); outs.append(it.get_tensor(o["index"])[:n])
    return np.concatenate(outs)

def latency_ms(path, threads, runs=50):
    it = TFLiteInterpreter(model_path=path, num_threads=threads); it.allocate_tensors()
    i = it.get_input_details()[0]; x = np.random.uniform(0, 255, (1, IMG_SIZE, IMG_SIZE, 3)).astype(np.float32)
    ts = []
    for k in range(runs + 5):
        t = time.perf_counter(); it.set_tensor(i["index"], x); it.invoke()
        if k >= 5:
            ts.append((time.perf_counter() - t) * 1000)
    return float(np.median(ts))

TFLITE = {}
for kind in ["float32", "float16", "int8"]:
    try:
        path = convert(kind); pt = tflite_predict(path, tf_batches(test_df))
        TFLITE[kind] = dict(path=path, size_mb=os.path.getsize(path) / 1e6, test=metrics(y_test, pt), pred=pt.argmax(1),
                            agreement_with_keras=float((pt.argmax(1) == test_pred).mean()),
                            max_prob_diff=float(np.abs(pt - p_test).max()),
                            ms_1_thread=latency_ms(path, 1), ms_4_threads=latency_ms(path, 4))
    except Exception as e:
        print(f"{kind} conversion failed: {e}")
display(pd.DataFrame({k: {kk: (v[kk] if not isinstance(v[kk], dict) else v[kk]["macro_f1"]) for kk in
                          ["size_mb", "test", "agreement_with_keras", "max_prob_diff", "ms_1_thread", "ms_4_threads"]}
                      for k, v in TFLITE.items()}).T.rename(columns={"test": "test_macro_f1"}).round(4))
good = [k for k, v in TFLITE.items() if v["test"]["macro_f1"] >= FINAL_TEST["macro_f1"] - 0.01 and v["agreement_with_keras"] >= 0.98]
RECOMMENDED = min(good, key=lambda k: TFLITE[k]["size_mb"]) if good else "float32"
print("Recommended phone model:", RECOMMENDED, "(latency measured on this notebook's CPU; phones differ)")

def pil_preprocess(path):
    # Independent re-implementation of the documented preprocessing (what an app developer would write).
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    w, h = im.size; s = min(w, h); l, t = (w - s) // 2, (h - s) // 2
    return np.asarray(im.crop((l, t, l + s, t + s)).resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR), np.float32)

paths = list(test_df["path"])
pil_batches = (np.stack([pil_preprocess(p) for p in paths[s:s + BATCH_TFL]]) for s in range(0, len(paths), BATCH_TFL))
pp = tflite_predict(TFLITE[RECOMMENDED]["path"], pil_batches)
PIL_CHECK = dict(agreement_with_tf_preprocessing=float((pp.argmax(1) == TFLITE[RECOMMENDED]["pred"]).mean()),
                 test=metrics(y_test, pp))
print(f"Pillow pipeline: top-1 agreement {PIL_CHECK['agreement_with_tf_preprocessing']:.3%} | "
      f"test macro-F1 {PIL_CHECK['test']['macro_f1']:.3f} (TensorFlow pipeline {TFLITE[RECOMMENDED]['test']['macro_f1']:.3f})")

## 13. Demo: the engine as the app uses it

`identify()` is the complete engine the app calls: it checks the input, prepares the photo, runs the phone model,
applies the calibration and threshold, and returns a plain-language answer with the top-3 guesses. Below it is
given valid photos of supported plants, photos of plants it has never seen, and invalid inputs (a file that is not
an image, a tiny image, a blank image and random noise).

In [ ]:
PHONE_MODEL = TFLiteInterpreter(model_path=TFLITE[RECOMMENDED]["path"], num_threads=4); PHONE_MODEL.allocate_tensors()

def identify(path):
    try:
        im = ImageOps.exif_transpose(Image.open(path)); im.load()
    except Exception:
        return dict(status="invalid", message="This file is not a readable image. Please take a photo of a plant.")
    if min(im.size) < 64:
        return dict(status="invalid", message="The image is too small. Please take a closer photo.")
    x = pil_preprocess(path)[None]
    i, o = PHONE_MODEL.get_input_details()[0], PHONE_MODEL.get_output_details()[0]
    PHONE_MODEL.set_tensor(i["index"], x); PHONE_MODEL.invoke()
    p = calibrate(PHONE_MODEL.get_tensor(o["index"]), TEMPERATURE)[0]
    top3 = [(CLASSES[LABELS[k]]["common"], float(p[k])) for k in np.argsort(-p)[:3]]
    k = int(p.argmax())
    if k == OTHER or p[k] < THRESHOLD:
        return dict(status="unknown", message="Not sure which plant this is. It may not be one of the 30 supported plants.", top3=top3)
    c = CLASSES[LABELS[k]]
    return dict(status="ok", message=f"{c['common']} ({c['scientific']}), {p[k]:.0%} confident", label=LABELS[k], top3=top3)

demo_dir = os.path.join(OUT_DIR, "demo_inputs"); os.makedirs(demo_dir, exist_ok=True)
open(os.path.join(demo_dir, "not_an_image.jpg"), "w").write("this is text, not a photo")
Image.new("RGB", (40, 40), "green").save(os.path.join(demo_dir, "tiny.jpg"))
Image.new("RGB", (640, 480), (225, 225, 225)).save(os.path.join(demo_dir, "blank.jpg"))   # plain grey wall
Image.fromarray(np.random.default_rng(0).integers(0, 255, (480, 640, 3), dtype=np.uint8)).save(os.path.join(demo_dir, "noise.jpg"))
demo = ([("valid", p) for p in test_df.sample(4, random_state=7)["path"]] +
        [("never-seen plant", p) for p in open_df.sample(min(2, len(open_df)), random_state=7)["path"]] +
        [("invalid", os.path.join(demo_dir, f)) for f in ["not_an_image.jpg", "tiny.jpg", "blank.jpg", "noise.jpg"]])
fig, axes = plt.subplots(2, 5, figsize=(20, 9))
for ax, (kind, path) in zip(axes.flat, demo):
    res = identify(path); ax.axis("off")
    try:
        ax.imshow(Image.open(path).convert("RGB"))
    except Exception:
        ax.text(0.5, 0.5, "(not an image)", ha="center", va="center")
    extra = "\n".join(f"{n[:20]} {s:.0%}" for n, s in res.get("top3", []))
    import textwrap
    ax.set_title(f"[{kind}] {res['status'].upper()}\n" + textwrap.fill(res["message"], 42) + f"\n{extra}", fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "figures", "demo.png"), dpi=100); plt.show()

## 14. Saving the deliverables

In [ ]:
final_dir = os.path.join(OUT_DIR, "final")
shutil.copyfile(TFLITE[RECOMMENDED]["path"], os.path.join(final_dir, "plantid_v1.tflite"))
shutil.copyfile(model_paths(FINAL_ARCH)[0], os.path.join(final_dir, "plantid_v1.keras"))
shutil.copyfile(os.path.join(DATA_ROOT, "labels.txt"), os.path.join(final_dir, "labels.txt"))
shutil.copyfile(os.path.join(DATA_ROOT, "classes.json"), os.path.join(final_dir, "classes.json"))
metadata = {
    "model_name": "PlantID v1", "architecture": FINAL_ARCH, "file": "plantid_v1.tflite", "tflite_variant": RECOMMENDED,
    "file_size_mb": round(TFLITE[RECOMMENDED]["size_mb"], 2),
    "input": {"name": "image", "shape": [1, IMG_SIZE, IMG_SIZE, 3], "dtype": "float32", "color_order": "RGB",
              "value_range": [0, 255], "preprocessing": ["apply EXIF orientation", "center-crop the largest square",
              f"resize to {IMG_SIZE}x{IMG_SIZE} (bilinear)", "raw RGB 0-255 as float32, NO normalisation (done inside the model)"]},
    "output": {"shape": [1, NUM_CLASSES], "dtype": "float32", "meaning": "softmax probabilities in labels.txt order"},
    "labels": LABELS, "other_plant_index": OTHER,
    "unknown_handling": {"temperature": TEMPERATURE, "threshold": THRESHOLD,
                         "rule": "p = softmax(log(probs)/temperature); if argmax(p)==other_plant_index or max(p)<threshold: 'not sure' (show top-3) else labels[argmax(p)]"},
    "test_metrics_keras": FINAL_TEST, "test_metrics_tflite": TFLITE[RECOMMENDED]["test"],
    "pillow_preprocessing_check": PIL_CHECK, "tensorflow": tf.__version__, "quick_run": QUICK_RUN,
}
json.dump(metadata, open(os.path.join(final_dir, "model_metadata.json"), "w"), indent=1)
report = dict(final_arch=FINAL_ARCH, comparison=RESULTS, cross_validation=CV, classification_report=REPORT, fit_check=FIT,
              most_confused=CONFUSED, unknown_handling=UNKNOWN, recommended_tflite=RECOMMENDED,
              tflite={k: {kk: v[kk] for kk in v if kk not in ("path", "pred")} for k, v in TFLITE.items()}, quick_run=QUICK_RUN)
json.dump(report, open(os.path.join(OUT_DIR, "evaluation_report.json"), "w"), indent=1, default=float)
shutil.make_archive(os.path.join(os.path.dirname(OUT_DIR.rstrip("/")), "plantid_outputs"), "zip", OUT_DIR)
for f in sorted(os.listdir(final_dir)):
    print(f"  {f:24s} {os.path.getsize(os.path.join(final_dir, f)) / 1e6:7.2f} MB")

## Conclusion and limitations

**What was built.** A 31-answer plant classifier (30 species + "other plant") trained with transfer learning,
compared against the base recipe, a CNN trained from scratch, a linear classifier and three phone networks, and
exported as a verified TensorFlow Lite file for the Android app. All scores above are measured on photos from
photographers the model never learned from; the comparison table, cross-validation and test report give the final numbers.

**Limitations**
* **Few photos from Kenya.** iNaturalist has under 100 photos per species from Kenya and no team photos were collected,
  so performance in Kenyan homes has not been measured directly.
* **Look-alike plants** (for example plants photographed without flowers, or seedlings) remain the most common errors;
  see the confusion matrix.
* **Mint** photos are mostly wild mint outdoors rather than potted kitchen mint, and **Ming aralia** has fewer photos (535).
* iNaturalist names are checked by the community, not by experts, so a small number of labels may be wrong.
* "Not sure" answers depend on a threshold: a stricter threshold rejects more unknown plants but also more real ones.
* Licences: most photos are CC BY-NC, suitable for this academic project but not for a commercial release.